# 03. Moreira and Muir's Table I on their sample: every published number beside the computed one (E1)

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Factor | the monthly return of one long-short portfolio that captures one source of common movement in stock returns: Mkt-RF, SMB, HML, Mom, RMW and CMA |
| Realised variance | the variance of a factor computed from the daily returns of one month; last month's is the signal |
| Volatility-managed factor | Moreira and Muir's construction: the factor's return times c divided by last month's realised variance, built in notebook 02 |
| Regression | the straight line through the months that best predicts one series from another, here the volatility-managed factor from the original factor, fitted by least squares (the line that makes the sum of squared misses smallest) |
| Beta | the slope of that line: how much the volatility-managed factor moves, on average, when the original factor moves by 1 |
| Alpha | the intercept of that line: the average monthly return of the volatility-managed factor that the original factor does not account for. Here it is a property of the regression and nothing more |
| Residual | the month's miss: the volatility-managed return minus the line's prediction |
| RMSE | the standard deviation of the residuals, with the two estimated coefficients taken into account (the sum of squared residuals divided by N - 2, square-rooted) |
| N | the number of months in the regression |
| Standard error | how far an estimate would typically move if the same months were drawn again; a small one means a precise estimate |
| White standard error | a standard error that allows the size of the residuals to change from month to month (White 1980). Moreira and Muir's Table I uses it |
| Newey-West standard error | a standard error that also allows each month's residual to be correlated with the previous months' (Newey and West 1987), here with 12 months |
| t-statistic | the estimate divided by its standard error |
| 5 per cent significance | a t-statistic further from 0 than 1.96. If the true alpha were 0, a t-statistic that far out would occur in 5 months out of 100 by chance |
| Appraisal ratio | alpha divided by the RMSE: the extra return per unit of the risk the original factor does not explain |
| Excess return | a return minus the one-month Treasury bill rate over the same period; every factor is one |
| Sharpe ratio | the average excess return divided by its standard deviation |
| Annualised | turned from a monthly figure into a yearly one: means times 12, standard deviations and Sharpe ratios times the square root of 12 |
| Verdict | met or not met: the result of an expectation, read from levels fixed before the code ran |
| Vintage | the version of French's files on the download date, named by the CRSP release they were built from |

**What this notebook does.** It builds the volatility-managed version of each of the six factors on Moreira and Muir's sample, regresses it on the original factor, and puts every number of their Table I (p. 1619) beside the computed one: beta, alpha, the standard error of alpha, N, R² and RMSE. It computes the appraisal ratio from both, and the market's volatility-managed Sharpe ratio for comparison with their Table V (p. 1626). It then judges E1.

**Why it comes at this point.** This is the paper's main table. A replication that matches it shows that the data, the signal and the construction of notebooks 01 and 02 are those of the paper; every later notebook builds on them. A replication that misses it stops the study until the miss is explained.

**The expectation it judges: E1**, fixed in `src/sq/constants.py` before this notebook's code was written:

1. On Moreira and Muir's sample (each factor's first month to April 2015), for at least five of the six factors:
    - the alpha's sign matches Table I's;
    - its 5 per cent significance category matches Table I's, with White standard errors, the paper's own;
    - the alpha lies within one published standard error of the published alpha;
    - the appraisal ratio, computed from Table I's alpha and RMSE with footnote 11's annualisation (p. 1620), lies within 0.05 of the paper's figure.
2. For the market, the volatility-managed Sharpe ratio also lies within 0.05 of 0.52 (Table V, Panel A), and the market counts as matching only if it does.

**Why these levels.** They are the replication criterion fixed on 6 September 2026, before the study's design: the sign and significance of the alpha, one standard error, and a margin of 0.05 on a ratio. French has revised his data since Moreira and Muir downloaded it, so exact equality is not expected; five of six lets one factor miss on a revised series, and a miss is reported, never chased. The paper prints a Sharpe ratio for the market only, so the third comparison for every factor is the appraisal ratio, which Table I gives through alpha and RMSE. The 5 per cent line is 1.96, from the normal distribution, as the paper's own stars use it.

**What I expect to see, and where it comes from.** E1 met. The published Table I, Panel A, for the six factors (alpha and its standard error in per cent a year, from monthly returns times 12):

| | Mkt-RF | SMB | HML | Mom | RMW | CMA |
|---|---|---|---|---|---|---|
| beta | 0.61 | 0.62 | 0.57 | 0.47 | 0.62 | 0.68 |
| alpha | 4.86 | -0.58 | 1.97 | 12.51 | 2.44 | 0.38 |
| standard error of alpha | 1.56 | 0.91 | 1.02 | 1.71 | 0.83 | 0.67 |
| t-statistic | 3.12 | -0.64 | 1.93 | 7.32 | 2.94 | 0.57 |
| N | 1,065 | 1,065 | 1,065 | 1,060 | 621 | 621 |
| R² | 0.37 | 0.38 | 0.32 | 0.22 | 0.38 | 0.46 |
| RMSE | 51.39 | 30.44 | 34.92 | 50.37 | 20.16 | 17.55 |
| appraisal ratio (alpha / RMSE times the square root of 12) | 0.33 | -0.07 | 0.20 | 0.86 | 0.42 | 0.08 |

Three things to watch. HML's published t-statistic, 1.93, sits 0.03 below the 1.96 line, so a small change in its alpha or standard error moves it across. SMB's and CMA's alphas are small next to their standard errors, so their sign can differ while the alpha still lies within one standard error. The N are known to match from notebook 01.

## How this notebook is organised

1. The modules are written into the runtime and the volatility-managed factors are built on Moreira and Muir's sample.
2. The regressions: the computed Table I.
3. One factor worked from monthly numbers to the paper's units.
4. The computed and the published numbers side by side, and E1 judged.
5. White and Newey-West standard errors compared.
6. The outputs, the summary and what could be wrong.

In [ ]:
import os
os.makedirs("src/sq", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
open("src/sq/__init__.py", "w").close()

In [ ]:
%%writefile src/sq/constants.py
"""
Pre-committed numbers for the study.

Every sample boundary, construction parameter, expectation and test setting is
defined here, one per name, with its source beside it. A notebook may read
these names; it may not introduce numbers of its own. The design was fixed on
8 November 2026 and restated against the three papers' printed tables on
10 November 2026, before any code that judges an expectation was written.

The three papers, with the journal's page numbers used throughout:

  MM    Moreira and Muir (2017), "Volatility-Managed Portfolios", Journal of
        Finance 72(4), 1611-1644.
  COWY Cederburg, O'Doherty, Wang and Yan (2020), "On the performance of
        volatility-managed portfolios", Journal of Financial Economics 138(1),
        95-117.
  BSC   Barroso and Santa-Clara (2015), "Momentum has its moments", Journal of
        Financial Economics 116(1), 111-120.

The Sharpe ratio test is Ledoit and Wolf (2008), "Robust performance
hypothesis testing with the Sharpe ratio", Journal of Empirical Finance 15(5),
850-859. The published numbers the expectations compare against are typed in
src/sq/published.py, never here.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":       "F-F_Research_Data_Factors_CSV.zip",             # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",       # the same, daily from 1926-07-01
    "factors5":       "F-F_Research_Data_5_Factors_2x3_CSV.zip",       # adds RMW and CMA, monthly from 1963-07
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # the same, daily from 1963-07-01
    "momentum":       "F-F_Momentum_Factor_CSV.zip",                   # Mom, monthly from 1927-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",             # Mom, daily from 1926-11-03
    "mom10":          "10_Portfolios_Prior_12_2_CSV.zip",              # ten portfolios formed on momentum, monthly from 1927-01
    "mom10_daily":    "10_Portfolios_Prior_12_2_Daily_CSV.zip",        # the same, daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# The six factors, by French's column names, and the file each comes from.
# Mkt-RF, SMB and HML come from the three-factor file, as in MM Table I and
# COWY section 2.1 (Fama and French 1993); RMW and CMA from the five-factor
# file (Fama and French 2015); Mom from the momentum file. RF is the
# three-factor file's one-month Treasury bill return.
FACTORS = ("Mkt-RF", "SMB", "HML", "Mom", "RMW", "CMA")
FACTOR_SOURCE = {
    "Mkt-RF": "factors3", "SMB": "factors3", "HML": "factors3",
    "Mom": "momentum", "RMW": "factors5", "CMA": "factors5",
}
RF_SOURCE = "factors3"

# WML, the momentum series of BSC: the top decile minus the bottom decile of
# the ten value-weighted portfolios formed on returns from month t-12 to t-2
# (BSC Appendix A, p. 119). Block 0 of both files holds the value-weighted returns.
WML_LONG = "Hi PRIOR"
WML_SHORT = "Lo PRIOR"
WML_BLOCK = 0
WML_BLOCK_TITLE_MUST_CONTAIN = "Value Weight"

# The end of the sample, fixed on 8 November 2026: August 2026 on any vintage.
# The loader cuts every file at this month, so that each notebook sees the same
# months on any later vintage; the vintage line and the SHA-256 in the
# provenance record say which revision of French's history a run used. A run on
# a later vintage, cut at August 2026, is a confirmation run: it is labelled by
# its vintage in the provenance record, it is expected to reproduce every
# verdict line and every printed decimal and not every digit, because French
# revises history, and it replaces no committed table without a dated decision.
SAMPLE_END = "2026-08"
SAMPLE_END_RULE_DATE = "2026-11-08"
CONFIRMATION_RUN = "a run on a later vintage, cut at August 2026, labelled by its vintage in the provenance record"

# French's daily files carry Saturday trading days until 24 May 1952. The number of
# trading days in a calendar month runs from 17 to 27 before June 1952 and from 15
# to 23 from 1953 on (counted on vintage 202609). Months with fewer trading days
# than the level below are flagged and reported; none is dropped.
LAST_SATURDAY_TRADING_DAY = "1952-05-24"
TRADING_DAYS_FLAG_BELOW = 15

PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

# ---------------------------------------------------------------------------
# 2. The samples
# ---------------------------------------------------------------------------

# The first month with a volatility-managed return, for each factor: the month
# after the first month of daily data, or the first month of the monthly file
# if that is later. These are COWY's start dates (Table 1 note, p. 99).
FACTOR_FIRST_MONTH = {
    "Mkt-RF": "1926-08", "SMB": "1926-08", "HML": "1926-08",
    "Mom": "1927-01", "RMW": "1963-08", "CMA": "1963-08",
}

# MM's sample. The paper says "1926 to 2015" (Table I note, p. 1619) and prints
# N = 1,065 for Mkt, SMB and HML, 1,060 for Mom and 621 for RMW and CMA. From
# the first months above, all three counts end in April 2015 exactly; an end in
# December 2015 would give 1,073, 1,068 and 629. The paper does not state the
# months, so the end is inferred from these counts; the computed N is reported
# beside the published N in every table that judges E1.
MM_SAMPLE_END = "2015-04"
MM_SAMPLE_END_IS_INFERRED = True
MM_EXPECTED_N = {"Mkt-RF": 1065, "SMB": 1065, "HML": 1065, "Mom": 1060, "RMW": 621, "CMA": 621}

# COWY's sample: the same first months to December 2016 (Table 1 note, p. 99).
COWY_SAMPLE_END = "2016-12"
COWY_EXPECTED_N = {"Mkt-RF": 1085, "SMB": 1085, "HML": 1085, "Mom": 1080, "RMW": 641, "CMA": 641}

# The months after MM's sample, the study's own question (E5).
AFTER_START = "2015-05"
AFTER_END = SAMPLE_END
AFTER_EXPECTED_MONTHS = 136

# BSC's sample is 1927:03 to 2011:12 (Table 3, p. 116). Their daily WML starts
# in August 1926 (Daniel and Moskowitz's data, Appendix A, p. 119); French's
# daily file of the ten momentum portfolios starts on 3 November 1926, so the
# 126 trading days the variance needs are first available at the end of April 1927
# and the first scaled month is May 1927, two months after the paper's.
BSC_PAPER_FIRST_MONTH = "1927-03"
BSC_FIRST_MONTH = "1927-05"
BSC_SAMPLE_END = "2011-12"
BSC_EXPECTED_MONTHS = 1016          # May 1927 to December 2011 inclusive
BSC_EXTENDED_END = SAMPLE_END       # the second period of scaled momentum: May 1927 to August 2026

# ---------------------------------------------------------------------------
# 3. The constructions
# ---------------------------------------------------------------------------

# The signal: last month's realised variance of the factor's daily returns.
# Each construction uses its own paper's definition.
#
# MM equation 2, p. 1616: each day's return minus the month's mean, squared and
# summed over the month. The equation is written for a 22-day month; the mean
# here is the month's own mean, the sum of its daily returns divided by its
# number of trading days.
RV_MM = "sum over the month's trading days of (daily return minus the month's mean daily return) squared"
# COWY equation 4, p. 98: 22 divided by the month's number of trading days J,
# times the sum of squared daily returns; no mean is removed.
RV_COWY_DAYS = 22
RV_COWY = "22 / J times the sum over the month's J trading days of the squared daily return"
# BSC equation 5, p. 115: 21 times the mean squared daily return over the last
# 126 trading days up to the month's last trading day, a monthly variance.
RV_BSC_TRADING_DAYS = 126
RV_BSC_DAYS_PER_MONTH = 21

# Which definition each comparison uses: a comparison with a paper's printed
# numbers uses that paper's definition. Fixed 10 November 2026.
RV_BY_COMPARISON = {
    "volatility-managed factor on MM's sample: E1, E4, E6 (Table IV, Table V)": "MM",
    "volatility-managed factor on COWY's sample: E2 (i), E6 (Table 1, Panel D)": "COWY",
    "real-time combination and real-time original: E2 (ii) to (iv), E5": "COWY",
    "volatility-managed factor after the sample, c fixed to April 2015: E5, no verdict": "MM",
    "scaled WML: E3": "BSC",
}

# The volatility-managed factor (MM equation 1, p. 1616): f_sigma_t = (c / RV_{t-1}) f_t,
# with c set so that the managed and the original factor have the same standard
# deviation over the sample the comparison runs on. c does not change a Sharpe
# ratio or an appraisal ratio (MM footnote 6, p. 1616; COWY p. 98).
#
# The real-time combination (COWY section 4.3, pp. 106-107, Table 5 Panel A):
# at the start of each month t after an initial training period of K months,
# on an expanding window of all months before t,
#   1. c_t gives the managed and the original factor the same variance over the window;
#   2. the means mu and the covariance Sigma of the managed and the original factor
#      are estimated over the window;
#   3. the weights are (x_sigma, x) = (1 / gamma) Sigma^-1 mu;
#   4. the position in the original factor is y_t = x_sigma c_t / RV_{t-1} + x, with |y_t| <= cap;
#   5. the month's return is y_t f_t.
# S2, the real-time original, is the same with the original factor alone:
# y_t = (1 / gamma) mu / sigma^2, under the same cap. S3, the ex post optimal
# combination, uses the moments of the whole out-of-sample period.
REALTIME_K = 120               # months; COWY p. 106, "K = 120 months"
REALTIME_GAMMA = 5.0           # risk aversion; COWY Table 5 note, p. 106
REALTIME_LEVERAGE_CAP = 5.0    # |y_t| <= 5; COWY Table 5 note, p. 106, and p. 107
REALTIME_WINDOW = "expanding"  # COWY p. 106

# Scaled momentum (BSC equation 6, p. 115): WML scaled by sigma_target / sigma_hat_{t-1},
# sigma_hat the square root of equation 5's variance, sigma_target an annualised
# volatility of 12 per cent (BSC p. 115), here as a monthly figure.
BSC_TARGET_VOL_ANNUAL = 0.12
BSC_TARGET_VOL_MONTHLY = BSC_TARGET_VOL_ANNUAL / 12 ** 0.5

# ---------------------------------------------------------------------------
# 4. Measures and tests (the decisions with defaults of 8 November 2026)
# ---------------------------------------------------------------------------

MONTHS_PER_YEAR = 12
# The regression f_sigma_t = alpha + beta f_t + e_t, monthly. Alpha is reported
# in per cent a year (times 12); the appraisal ratio is alpha / sd(e) times the
# square root of 12 (MM footnote 11, p. 1620); Sharpe ratios are annualised by
# the square root of 12.
ALPHA_ANNUALISE = MONTHS_PER_YEAR
SHARPE_ANNUALISE = MONTHS_PER_YEAR ** 0.5

# Standard errors. E1 is judged with heteroskedasticity-robust standard errors,
# the papers' own (MM Table I note, p. 1619; COWY Table 3 note, White 1980,
# p. 104). The Newey and West standard error at 12 lags is reported beside it
# and is the one E4 to E6 use. HML's published alpha, 1.97 with a standard
# error of 1.02, has a t-statistic of 1.93 against the 1.96 line, so the
# choice can move a factor across it.
SE_E1 = "HC0"                  # White (1980)
NEWEY_WEST_LAGS = 12
SIGNIFICANCE_LEVEL = 0.05

# The Sharpe ratio difference (Ledoit and Wolf 2008): the studentised circular
# block bootstrap on paired series.
BOOTSTRAP_BLOCK_MONTHS = 12
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_LEVEL = 0.95
BOOTSTRAP_SEED = 20261108      # the date of the design, as a number

# The size check, run before the test touches a factor: 1,000 simulated pairs
# with equal Sharpe ratios and the sample's length, Student-t shocks with five
# degrees of freedom, a common component with the market's correlation between
# the pair, no difference in means.
SIZE_CHECK_PAIRS = 1000
SIZE_CHECK_T_DOF = 5
SIZE_CHECK_SEED = 20261110

# Costs: one proportional cost per unit of one-way turnover, the turnover of a
# volatility-managed strategy in a month being the absolute change in its
# position, |w_t - w_{t-1}|. Factor portfolios are traded through liquid
# large-cap long-short books, for which 10 basis points per unit of notional
# traded is the order the literature on implementation costs reports (MM Table
# IV, p. 1625, takes 10 basis points from Frazzini, Israel and Moskowitz); the
# one sensitivity doubles it.
COST_PER_TURNOVER = 0.0010
COST_SENSITIVITY = 0.0020

# ---------------------------------------------------------------------------
# 5. The specifications, counted before the code runs
# ---------------------------------------------------------------------------

# A specification is one pair of return series compared on one sample. The
# list is written out so that nothing can be added after a run without a dated
# disclosure. Cost levels are sensitivities of the same specifications and are
# not counted again.
_SAMPLES = {
    "MM": "first month to April 2015",
    "COWY": "first month to December 2016",
    "AFTER": "May 2015 to August 2026",
}
SPECIFICATIONS = (
    tuple(("E1", f, "MM", "volatility-managed", "original") for f in FACTORS)
    + tuple(("E2 (i)", f, "COWY", "volatility-managed", "original") for f in FACTORS)
    + tuple(("E2 (ii, iii)", f, "COWY", "real-time combination S1", "real-time original S2") for f in FACTORS)
    + tuple(("E2 (iv)", f, "COWY", "real-time combination S1", "ex post optimal combination S3") for f in FACTORS)
    + tuple(("E5", f, "AFTER", "real-time combination S1", "real-time original S2") for f in FACTORS)
    + tuple(("E5, no verdict", f, "AFTER", "volatility-managed, c to April 2015", "original") for f in FACTORS)
    + (("E3", "WML", "BSC 1927-05 to 2011-12", "scaled WML", "WML"),
       ("E3, extended", "WML", "BSC 1927-05 to 2026-08", "scaled WML", "WML"))
)
SPECIFICATION_COUNT = 38
SPECIFICATION_COUNT_DATE = "2026-11-10"

# ---------------------------------------------------------------------------
# 6. The expectations, written before the code
# ---------------------------------------------------------------------------

# Each is a strict pass or fail, reported in the same voice whether met or
# not; no level moves after a run. The levels the code reads are the named
# constants; the text and the reason are what each notebook quotes.

E1_MIN_FACTORS = 5
E1_ALPHA_WITHIN_SE = 1.0
E1_APPRAISAL_BAND = 0.05
E1_MARKET_SHARPE_BAND = 0.05

E2_SHARPE_BAND = 0.05
E2_SIGN_MIN_FACTORS = 5
E2_ONLY_SIGNIFICANT = ("Mom",)
E2_S1_S3_BAND = 0.05

E3_SHARPE_BAND = 0.05
E3_DRAWDOWN_BAND = 0.05        # five percentage points of drawdown

E4_SIZE_RANGE = (0.035, 0.065)
E4_MIN_FACTORS = 5

E5_MAX_FACTORS_ABOVE = 2

E6_COST = COST_PER_TURNOVER
E6_MIN_NW_T = 2.0
E6_MARKET_PERCENTILES = (50, 75, 90, 99)
E6_PERCENTILE_BAND = 0.15      # P50, P75 and P90
E6_P99_BAND = 1.0

EXPECTATIONS = {
    "E1": {
        "date": "2026-11-10",
        "text": (
            "On MM's sample (each factor's first month to April 2015), for at least five of the six factors: "
            "the alpha's sign and its 5 per cent significance category, with heteroskedasticity-robust (White) "
            "standard errors, match Table I (p. 1619); the alpha lies within one published standard error of the "
            "published alpha; and the appraisal ratio, computed from Table I's alpha and RMSE with footnote 11's "
            "annualisation (p. 1620), lies within 0.05 of the paper's figure, the market's 0.33 from the table, with "
            "the text's 0.34 (p. 1620) and 0.33 (p. 1612) noted beside it. For the market, the volatility-managed "
            "Sharpe ratio also lies within 0.05 of 0.52 (Table V, Panel A, p. 1626), and the market counts as "
            "matching only if it does. Every factor's comparisons are reported, whatever the count, with the "
            "computed N beside the published N."
        ),
        "reason": (
            "The replication criterion fixed on 6 September 2026: sign and significance, one standard error, and a "
            "band of 0.05. Its Sharpe ratio leg is the appraisal ratio for every factor because the paper prints no "
            "Sharpe ratio for any factor but the market. French revises history, so exact equality is not expected; "
            "five of six allows one factor to miss on a revised series, and the miss is reported, not chased."
        ),
    },
    "E2": {
        "date": "2026-11-10",
        "text": (
            "On COWY's sample (August 1926, January 1927 or August 1963 to December 2016), for each of the six "
            "factors: (i) the original and the volatility-managed Sharpe ratios lie within 0.05 of their Table 1 "
            "(p. 99); (ii) the real-time combination's Sharpe ratio S1 and the real-time original's S2 lie within "
            "0.05 of Table 5, Panel A (p. 106); (iii) S1 minus S2 has the published sign for at least five of the "
            "six, and the only difference significant at 5 per cent by the Jobson and Korkie test is momentum's, "
            "as published; (iv) S1 minus S3, the ex post optimal combination, lies within 0.05 of the published "
            "figure for all six. All twenty-four comparisons are reported, whatever the count."
        ),
        "reason": (
            "The critique's baseline is a real-time combination of the volatility-managed and the original factor "
            "(section 4.3, pp. 106-107): the scaling constant alone does not move a Sharpe ratio (MM footnote 6, "
            "p. 1616; COWY p. 98), and the weights of the combination are what an investor cannot know in advance."
        ),
    },
    "E3": {
        "date": "2026-11-10",
        "text": (
            "On May 1927 to December 2011, the Sharpe ratios of unscaled and scaled WML lie within 0.05 of 0.53 and "
            "0.97 (BSC Table 3, p. 116), and the maximum drawdowns lie within five percentage points of 96.69 and "
            "45.20 per cent (p. 112). The first scaled month is two months after the paper's 1927:03 because "
            "French's daily file of the ten momentum portfolios starts on 3 November 1926."
        ),
        "reason": (
            "The paper's two headline results, on its own momentum series (Appendix A, p. 119); the Sharpe band is "
            "the one used throughout the study."
        ),
    },
    "E4": {
        "date": "2026-11-08",
        "text": (
            "The size check rejects between 3.5 and 6.5 per cent of the 1,000 simulated null pairs at the 5 per cent "
            "level; and on MM's sample the bootstrap's verdict on the Sharpe ratio difference agrees in sign and in "
            "significance with the Newey and West t-statistic on the alpha for at least five of the six factors."
        ),
        "reason": (
            "A test that is wrong about its own size is not used; the band is two standard errors of a proportion "
            "of 0.05 over 1,000 draws. The alpha and the Sharpe ratio difference are two readings of the same gain "
            "and should agree."
        ),
    },
    "E5": {
        "date": "2026-11-10",
        "text": (
            "On the months after MM's sample, May 2015 to August 2026 (136 months), with the paper's c fixed on the "
            "data to April 2015 and the real-time combination carried forward from its expanding window: for each "
            "factor, the Sharpe ratio of the real-time combination against the real-time original S2, as in COWY "
            "Table 5, and of the volatility-managed factor against the original factor itself, each difference with "
            "its Ledoit and Wolf bootstrap interval. On at most two of the six factors the real-time combination's "
            "Sharpe ratio exceeds S2's by more than the interval allows, and the median across the six of the Sharpe "
            "difference lies within plus or minus the median of the six intervals' half-widths. The same two counts "
            "are reported for the volatility-managed factor, in no verdict."
        ),
        "reason": (
            "The study's own question, with the prior the critique gives: gains an investor could not have set in "
            "real time should show little in months neither paper could have known. If more than two factors clear "
            "the interval, the expectation fails and the finding is reported as such."
        ),
    },
    "E6": {
        "date": "2026-11-10",
        "text": (
            "At 10 basis points per unit of turnover, the market's volatility-managed alpha over MM's sample stays "
            "positive and more than two Newey and West standard errors from zero (the paper: 3.98 per cent a year at "
            "10 basis points, an average absolute change in weight of 0.73 and a break-even cost of 56 basis points, "
            "Table IV, p. 1625). On MM's sample, the market's P50, P75 and P90 of the leverage lie within 0.15 of "
            "0.93, 1.59 and 2.64, and its P99 within 1.0 of 6.39 (Table V, Panel A, p. 1626); on COWY's sample, each "
            "factor's P50 lies within 0.15 of their Table 1, Panel D (p. 99), and its P99 within 1.0 of theirs (6.47, "
            "5.07, 5.89, 8.64, 5.02, 4.56)."
        ),
        "reason": (
            "MM report that the market strategy's gain survives moderate costs (Table IV) and print the distribution "
            "of its leverage (Table V); COWY print the 1st, 50th and 99th percentiles for every factor. The "
            "expectation says whether both hold on this vintage."
        ),
    },
}

In [ ]:
%%writefile src/sq/published.py
"""
The published numbers of the three papers the study replicates, transcribed
from the journal PDFs, with the page and the table of every number.

The comparison tables in the notebooks are built from this module, never typed
into a notebook. tests/test_published.py re-reads every number from the
journal PDFs when they are present on the machine (they are copyrighted and
live outside the repository) and fails on any digit that differs.

Units are the papers' own: returns, alphas and standard deviations in per cent
a year, Sharpe and appraisal ratios annualised, leverage as a multiple of the
original factor. The keys are French's column names for the six factors
(MM print Mkt, SMB, HML, Mom, RMW, CMA; COWY print MKT, SMB, HML, MOM, RMW, CMA).
"""
from __future__ import annotations

import math

FACTORS = ("Mkt-RF", "SMB", "HML", "Mom", "RMW", "CMA")

# ---------------------------------------------------------------------------
# Moreira and Muir (2017), Journal of Finance 72(4), 1611-1644
# ---------------------------------------------------------------------------

# Table I, Panel A, p. 1619: f_sigma = alpha + beta f + e, univariate, monthly
# data with alpha and RMSE multiplied by 12; standard errors adjust for
# heteroskedasticity. The table's first six of ten columns.
MM_TABLE_I = {
    #          beta  (se)   alpha  (se)     N     R2    RMSE
    "Mkt-RF": (0.61, 0.05,  4.86, 1.56, 1065, 0.37, 51.39),
    "SMB":    (0.62, 0.08, -0.58, 0.91, 1065, 0.38, 30.44),
    "HML":    (0.57, 0.07,  1.97, 1.02, 1065, 0.32, 34.92),
    "Mom":    (0.47, 0.07, 12.51, 1.71, 1060, 0.22, 50.37),
    "RMW":    (0.62, 0.08,  2.44, 0.83,  621, 0.38, 20.16),
    "CMA":    (0.68, 0.05,  0.38, 0.67,  621, 0.46, 17.55),
}
MM_TABLE_I_FIELDS = ("beta", "beta_se", "alpha", "alpha_se", "N", "R2", "RMSE")

# Appraisal ratios printed in the text: the market's 0.34 and HML's 0.20 and
# profitability's 0.41 (p. 1620), and the market's 0.33 in the introduction
# (p. 1612). The figures E1 compares against are computed from Table I below.
MM_TEXT_APPRAISAL_P1620 = {"Mkt-RF": 0.34, "HML": 0.20, "RMW": 0.41}
MM_TEXT_APPRAISAL_P1612 = {"Mkt-RF": 0.33}

# Table IV, p. 1625: the market, "Realized variance" row. Average absolute
# change in monthly weight, expected return and alpha before costs (per cent a
# year), alpha after 1, 10 and 14 basis points of trading cost, and the
# break-even cost in basis points.
MM_TABLE_IV_MARKET = {
    "mean_abs_change_in_weight": 0.73,
    "expected_return": 9.47,
    "alpha": 4.86,
    "alpha_1bp": 4.77,
    "alpha_10bp": 3.98,
    "alpha_14bp": 3.63,
    "break_even_bp": 56,
}

# Table V, Panel A, p. 1626: the market, "Realized variance" row. Alpha and its
# standard error, Sharpe ratio, appraisal ratio, and percentiles of the weight
# c / RV.
MM_TABLE_V_MARKET = {
    "alpha": 4.86, "alpha_se": 1.56, "sharpe": 0.52, "appraisal": 0.34,
    "P50": 0.93, "P75": 1.59, "P90": 2.64, "P99": 6.39,
}


def mm_appraisal_from_table_i(factor: str) -> float:
    """alpha / RMSE times the square root of 12, from Table I (footnote 11, p. 1620)."""
    beta, beta_se, alpha, alpha_se, n, r2, rmse = MM_TABLE_I[factor]
    return alpha / rmse * math.sqrt(12)


# ---------------------------------------------------------------------------
# Cederburg, O'Doherty, Wang and Yan (2020), Journal of Financial Economics
# 138(1), 95-117. Their sample ends in December 2016 (Table 1 note, p. 99).
# ---------------------------------------------------------------------------

# Table 1, p. 99. Panel A, original factors, and Panel B, volatility-managed
# factors: mean, standard deviation (per cent a year) and Sharpe ratio.
COWY_TABLE_1_ORIGINAL = {
    #          mean    sd     SR
    "Mkt-RF": (7.80, 18.61, 0.42),
    "SMB":    (2.57, 11.12, 0.23),
    "HML":    (4.84, 12.14, 0.40),
    "Mom":    (7.94, 16.39, 0.48),
    "RMW":    (2.92,  7.71, 0.38),
    "CMA":    (3.72,  6.97, 0.53),
}
COWY_TABLE_1_MANAGED = {
    "Mkt-RF": (9.55, 18.61, 0.51),
    "SMB":    (0.86, 11.12, 0.08),
    "HML":    (4.64, 12.14, 0.38),
    "Mom":    (16.17, 16.39, 0.99),
    "RMW":    (3.94,  7.71, 0.51),
    "CMA":    (2.79,  6.97, 0.40),
}
# Panel C: Sharpe ratio difference, managed minus original, and the Jobson and
# Korkie p-value. Panel D: correlation with the original factor and the 1st,
# 50th and 99th percentiles of the weight c* / RV.
COWY_TABLE_1_DIFFERENCE = {
    "Mkt-RF": (0.09, 0.30), "SMB": (-0.15, 0.09), "HML": (-0.02, 0.86),
    "Mom": (0.50, 0.00), "RMW": (0.13, 0.29), "CMA": (-0.13, 0.23),
}
COWY_TABLE_1_PANEL_D = {
    #          corr   P01   P50   P99
    "Mkt-RF": (0.63, 0.04, 0.96, 6.47),
    "SMB":    (0.63, 0.03, 0.81, 5.07),
    "HML":    (0.57, 0.04, 1.02, 5.89),
    "Mom":    (0.48, 0.04, 1.01, 8.64),
    "RMW":    (0.59, 0.04, 1.11, 5.02),
    "CMA":    (0.68, 0.06, 0.97, 4.56),
}

# Table 3, Panel A.1, p. 104: the full-sample spanning regression. Alpha (per
# cent a year) with its t-statistic from White (1980) standard errors, beta with
# its t-statistic, adjusted R2, appraisal ratio; Panel A.2, the scaling constant c*.
COWY_TABLE_3 = {
    #          alpha  (t)    beta  (t)     R2    AR     c*
    "Mkt-RF": (4.63,  3.08, 0.63, 11.32, 0.40,  0.32, 10.33),
    "SMB":    (-0.76, -0.87, 0.63,  7.75, 0.40, -0.09,  2.63),
    "HML":    (1.87,  1.88, 0.57,  7.65, 0.33,  0.19,  2.95),
    "Mom":    (12.39, 7.31, 0.48,  7.13, 0.23,  0.86,  4.60),
    "RMW":    (2.23,  2.57, 0.59,  7.10, 0.34,  0.36,  1.48),
    "CMA":    (0.26,  0.39, 0.68, 13.82, 0.46,  0.05,  1.53),
}
COWY_TABLE_3_FIELDS = ("alpha", "alpha_t", "beta", "beta_t", "R2", "AR", "c")

# Table 5, Panel A, p. 106: the real-time strategies, K = 120, expanding
# window, gamma = 5, leverage at most 5; Sharpe ratios over the out-of-sample
# period, p-values of the differences in the second tuple (Jobson and Korkie).
COWY_TABLE_5 = {
    #          S1    S2    S1-S2  S3    S1-S3
    "Mkt-RF": (0.42, 0.46, -0.04, 0.53, -0.11),
    "SMB":    (0.14, 0.19, -0.06, 0.26, -0.12),
    "HML":    (0.38, 0.43, -0.06, 0.50, -0.12),
    "Mom":    (0.92, 0.49,  0.44, 0.99, -0.07),
    "RMW":    (0.44, 0.31,  0.13, 0.58, -0.14),
    "CMA":    (0.52, 0.56, -0.03, 0.64, -0.11),
}
COWY_TABLE_5_P = {
    #          S1-S2  S1-S3
    "Mkt-RF": (0.64, 0.01),
    "SMB":    (0.37, 0.14),
    "HML":    (0.41, 0.08),
    "Mom":    (0.00, 0.07),
    "RMW":    (0.53, 0.37),
    "CMA":    (0.20, 0.00),
}

# ---------------------------------------------------------------------------
# Barroso and Santa-Clara (2015), Journal of Financial Economics 116(1), 111-120.
# Sample 1927:03 to 2011:12.
# ---------------------------------------------------------------------------

BSC_FIELDS = ("max", "min", "mean", "sd", "kurtosis", "skewness", "sharpe")
# Table 1, p. 113, and Table 3, p. 116: monthly returns in per cent, mean and
# standard deviation annualised, excess kurtosis, skewness, annualised Sharpe ratio.
BSC_TABLE_1_WML = (26.18, -78.96, 14.46, 27.53, 18.24, -2.47, 0.53)
BSC_TABLE_3_WML = (26.18, -78.96, 14.46, 27.53, 18.24, -2.47, 0.53)
BSC_TABLE_3_WML_SCALED = (21.95, -28.40, 16.50, 16.95, 2.68, -0.42, 0.97)
BSC_TABLE_3_INFORMATION_RATIO = 0.78
# p. 112: maximum drawdown in per cent, unscaled and scaled.
BSC_MAX_DRAWDOWN = {"WML": -96.69, "WML scaled": -45.20}


def table(name: str):
    """One published table as a pandas DataFrame, factors as rows."""
    import pandas as pd
    if name == "mm_table_i":
        df = pd.DataFrame(MM_TABLE_I, index=MM_TABLE_I_FIELDS).T
        df["appraisal"] = [mm_appraisal_from_table_i(f) for f in df.index]
        return df
    if name == "cowy_table_1":
        o = pd.DataFrame(COWY_TABLE_1_ORIGINAL, index=["mean", "sd", "sharpe"]).T
        m = pd.DataFrame(COWY_TABLE_1_MANAGED, index=["mean", "sd", "sharpe"]).T
        d = pd.DataFrame(COWY_TABLE_1_DIFFERENCE, index=["difference", "p"]).T
        q = pd.DataFrame(COWY_TABLE_1_PANEL_D, index=["corr", "P01", "P50", "P99"]).T
        return pd.concat({"original": o, "managed": m, "difference": d, "weights": q}, axis=1)
    if name == "cowy_table_3":
        return pd.DataFrame(COWY_TABLE_3, index=COWY_TABLE_3_FIELDS).T
    if name == "cowy_table_5":
        s = pd.DataFrame(COWY_TABLE_5, index=["S1", "S2", "S1-S2", "S3", "S1-S3"]).T
        p = pd.DataFrame(COWY_TABLE_5_P, index=["p S1-S2", "p S1-S3"]).T
        return pd.concat([s, p], axis=1)
    if name == "bsc_table_3":
        return pd.DataFrame({"WML": BSC_TABLE_3_WML, "WML scaled": BSC_TABLE_3_WML_SCALED}, index=BSC_FIELDS).T
    raise KeyError(name)

In [ ]:
%%writefile src/sq/french_loader.py
"""
Ken French data library: download, record, parse, assemble.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with one or more blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: eight digits
for daily data (19260701), six for monthly (192607), four for annual (1927).
Missing values are coded -99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a Period (day, month
or year), in decimal returns, with the missing codes replaced by NaN. It also
writes a provenance record for every file it downloads: the URL, the download
time, the SHA-256 of the zip, and the CRSP vintage line French prints at the
top of each CSV. A result is only as reproducible as the vintage it was
computed on, and French revises history, so the record is part of every output.

Two facts about the daily files matter for a monthly variance built from them.
They carry Saturday trading days until 24 May 1952, when the New York Stock
Exchange stopped trading on Saturdays. The number of trading days in a calendar
month therefore runs from 17 to 27 before June 1952 and from 15 to 23 from
1953 on (counted on vintage 202609). The variance definitions in constants.py
say how each paper treats the number of trading days.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{8}|\d{6}|\d{4})\s*,")   # daily YYYYMMDD, monthly YYYYMM, annual YYYY


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "daily" | "monthly" |
    "annual", "frame": DataFrame} where the frame is indexed by Period (day,
    month or year) and holds floats in the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 8:
                idx = pd.PeriodIndex(pd.to_datetime(raw.index.astype(str), format="%Y%m%d"), freq="D")
                freq = "daily"
            elif date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One block of one file, in decimal returns with missing codes as NaN.

    The name dates from the monthly files; the function serves the daily files
    in the same way (the block's index is then a daily Period).

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = cut_to_sample_end(chosen["frame"].copy())
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def cut_to_sample_end(frame: pd.DataFrame, end: str | None = None) -> pd.DataFrame:
    """The frame up to the end of the sample (constants.SAMPLE_END): a monthly block to that
    month, a daily block to the last day of that month, an annual block to that year. A later vintage of French's
    files carries months after the end, which the study does not use; the vintage in the provenance record still says
    which revision of history the months before the end came from. With no end set, the frame is returned as it is."""
    end = C.SAMPLE_END if end is None else end
    if end is None or not isinstance(frame.index, pd.PeriodIndex) or len(frame) == 0:
        return frame
    end_m = pd.Period(end, "M")
    freq = frame.index.freqstr or ""
    if freq.startswith("M"):
        limit = end_m
    elif freq.startswith("D"):
        limit = end_m.asfreq("D", "end")
    else:
        limit = end_m.asfreq(frame.index.freq, "end")
    return frame.loc[:limit]


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# The study's series, assembled from the files above
# ---------------------------------------------------------------------------

def _suffix(freq: str) -> str:
    if freq not in ("monthly", "daily"):
        raise ValueError(f"freq is 'monthly' or 'daily', not {freq!r}")
    return "" if freq == "monthly" else "_daily"


def factors(freq: str = "monthly", cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """The six factors of constants.FACTORS and the risk-free rate, in decimal returns, cut at the sample's end.

    Each factor comes from the file constants.FACTOR_SOURCE names; the columns are aligned on the union of the
    dates, so a factor whose file starts later (momentum, RMW and CMA) is NaN before its first date.
    """
    sfx = _suffix(freq)
    cols = {}
    frames = {}
    for f in C.FACTORS + ("RF",):
        key = (C.FACTOR_SOURCE[f] if f != "RF" else C.RF_SOURCE) + sfx
        if key not in frames:
            frames[key] = load_monthly(key, 0, cache_dir)
        cols[f] = frames[key][f]
    out = pd.DataFrame(cols)
    out.attrs["provenance"] = [frames[k].attrs["provenance"] for k in sorted(frames)]
    return out


def wml(freq: str = "monthly", cache_dir: str | Path = "data/french") -> pd.Series:
    """WML: the top decile minus the bottom decile of the ten value-weighted portfolios formed on momentum,
    in decimal returns, cut at the sample's end (Barroso and Santa-Clara, Appendix A)."""
    key = "mom10" + _suffix(freq)
    frame = load_monthly(key, C.WML_BLOCK, cache_dir)
    if C.WML_BLOCK_TITLE_MUST_CONTAIN not in frame.attrs["title"]:
        raise ValueError(f"block {C.WML_BLOCK} of {key} is {frame.attrs['title']!r}, not the value-weighted returns")
    out = (frame[C.WML_LONG] - frame[C.WML_SHORT]).rename("WML")
    out.attrs["provenance"] = frame.attrs["provenance"]
    return out


def trading_days_per_month(daily: pd.DataFrame | pd.Series) -> pd.Series:
    """The number of daily rows in each calendar month, indexed by month."""
    idx = daily.index
    months = idx.asfreq("M") if isinstance(idx, pd.PeriodIndex) else pd.PeriodIndex(idx, freq="M")
    return pd.Series(1, index=months).groupby(level=0).sum().rename("trading days")

In [ ]:
%%writefile src/sq/managed.py
"""
The signal and the two constructions built on it.

The signal is last month's realised variance of a factor, computed from the
factor's daily returns. Each paper defines it in its own way, and each
comparison with a paper's printed numbers uses that paper's definition
(constants.RV_BY_COMPARISON):

  rv_mm     Moreira and Muir, equation 2, p. 1616: each day's return minus the
            month's mean daily return, squared and summed over the month.
  rv_cowy   Cederburg, O'Doherty, Wang and Yan, equation 4, p. 98: 22 divided
            by the month's number of trading days J, times the sum of squared
            daily returns.
  rv_bsc    Barroso and Santa-Clara, equation 5, p. 115: 21 times the mean
            squared daily return over the last 126 trading days up to the
            month's last trading day.

The volatility-managed factor (Moreira and Muir, equation 1, p. 1616) holds
c / RV_{t-1} units of the factor in month t, with c set so that the managed
and the original factor have the same standard deviation over the sample.

The real-time combination (Cederburg et al., section 4.3, pp. 106-107) mixes
the volatility-managed and the original factor with mean-variance weights
estimated each month from the months before it only. Its position in the
original factor is y_t = x_sigma c_t / RV_{t-1} + x, truncated at plus or
minus the cap (p. 107: "a leverage constraint of |y_t| <= 5"). The ex post
optimal combination (Table 5, p. 106) holds one pair of weights, set with the
means and the covariance of the same months it is applied to.

All returns are monthly decimal excess returns indexed by a monthly Period;
daily returns are indexed by a daily Period.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C


# ---------------------------------------------------------------------------
# The signal
# ---------------------------------------------------------------------------

def _by_month(daily: pd.Series):
    daily = daily.dropna()
    return daily.groupby(daily.index.asfreq("M"))


def rv_mm(daily: pd.Series) -> pd.Series:
    """Moreira and Muir's realised variance of each calendar month (equation 2)."""
    return _by_month(daily).apply(lambda r: float(((r - r.mean()) ** 2).sum())).rename("rv_mm")


def rv_cowy(daily: pd.Series) -> pd.Series:
    """Cederburg et al.'s realised variance of each calendar month (equation 4): 22 / J times the sum of squares."""
    g = _by_month(daily)
    return (C.RV_COWY_DAYS * g.apply(lambda r: float((r ** 2).sum())) / g.size()).rename("rv_cowy")


def rv_bsc(daily: pd.Series) -> pd.Series:
    """Barroso and Santa-Clara's variance at each month's last trading day (equation 5): 21 times the mean
    squared daily return over the last 126 trading days. NaN until 126 trading days are available."""
    daily = daily.dropna()
    mean_sq = (daily ** 2).rolling(C.RV_BSC_TRADING_DAYS, min_periods=C.RV_BSC_TRADING_DAYS).mean()
    month_end = mean_sq.groupby(mean_sq.index.asfreq("M")).last()
    return (C.RV_BSC_DAYS_PER_MONTH * month_end).rename("rv_bsc")


def lagged(rv: pd.Series) -> pd.Series:
    """The variance of month t-1 indexed at month t, the value a position for month t can use."""
    out = rv.copy()
    out.index = out.index + 1
    return out.rename(f"{rv.name}_lag")


# ---------------------------------------------------------------------------
# The volatility-managed factor
# ---------------------------------------------------------------------------

def volatility_managed(f: pd.Series, rv: pd.Series, start: str, end: str, c: float | None = None):
    """The volatility-managed factor on the months start to end.

    Returns (frame, c). The frame has one row per month with the factor's return f, last month's variance
    rv_lag, the position weight = c / rv_lag and the managed return = weight * f. With c None, c is set so
    that the managed and the original return have the same standard deviation over these months (Moreira
    and Muir, p. 1616); a given c is used as it is, for months after the sample c was set on.
    """
    df = pd.DataFrame({"f": f, "rv_lag": lagged(rv)}).loc[start:end].dropna()
    raw = df["f"] / df["rv_lag"]
    if c is None:
        c = float(df["f"].std() / raw.std())
    df["weight"] = c / df["rv_lag"]
    df["managed"] = df["weight"] * df["f"]
    return df, c


# ---------------------------------------------------------------------------
# The real-time combination
# ---------------------------------------------------------------------------

def mean_variance_weights(returns: np.ndarray, gamma: float) -> np.ndarray:
    """(1 / gamma) Sigma^-1 mu for the columns of `returns` (months in rows), with the sample covariance."""
    mu = returns.mean(axis=0)
    sigma = np.atleast_2d(np.cov(returns, rowvar=False))
    return np.linalg.solve(sigma, mu) / gamma


def real_time(f: pd.Series, rv: pd.Series, first: str, end: str, k: int = C.REALTIME_K,
              gamma: float = C.REALTIME_GAMMA, cap: float = C.REALTIME_LEVERAGE_CAP) -> pd.DataFrame:
    """Cederburg et al.'s real-time combination (S1) and real-time original (S2), month by month.

    The months run from `first` to `end`; the first k are the initial training period and carry no
    position. For each later month t the window is every month from `first` to t-1 (an expanding window):

      1. c_t gives the managed and the original factor the same variance over the window;
      2. the means and the covariance of the managed and the original factor over the window give the
         weights (x_sigma, x) = (1 / gamma) Sigma^-1 mu;
      3. the position in the original factor is y_t = x_sigma c_t / RV_{t-1} + x, truncated at +-cap;
      4. S1's return is y_t f_t.

    S2 holds y2_t = (1 / gamma) mu_f / sigma_f^2 of the original factor alone, under the same cap, and
    returns y2_t f_t. Nothing from month t or later enters month t's positions.
    """
    df = pd.DataFrame({"f": f, "rv_lag": lagged(rv)}).loc[first:end].dropna()
    fr = df["f"].to_numpy()
    rv_lag = df["rv_lag"].to_numpy()
    raw = fr / rv_lag
    rows = []
    for i in range(k, len(df)):
        f_win, raw_win = fr[:i], raw[:i]
        c_t = f_win.std(ddof=1) / raw_win.std(ddof=1)
        x_sigma, x = mean_variance_weights(np.column_stack([c_t * raw_win, f_win]), gamma)
        y = x_sigma * c_t / rv_lag[i] + x
        y_capped = float(np.clip(y, -cap, cap))
        y2 = float(np.clip(f_win.mean() / (gamma * f_win.var(ddof=1)), -cap, cap))
        rows.append({
            "month": df.index[i], "window": i, "c": c_t, "x_sigma": x_sigma, "x": x,
            "y_uncapped": y, "y": y_capped, "at_cap": abs(y) > cap,
            "f": fr[i], "s1": y_capped * fr[i], "y2": y2, "s2": y2 * fr[i],
        })
    out = pd.DataFrame(rows).set_index("month")
    out.index = pd.PeriodIndex(out.index, freq="M")
    return out


def ex_post_combination(managed: pd.Series, original: pd.Series, gamma: float = C.REALTIME_GAMMA):
    """Cederburg et al.'s ex post optimal combination, S3 (Table 5, p. 106): one pair of weights
    (x_sigma, x) = (1 / gamma) Sigma^-1 mu from the means and the covariance of the managed and the
    original factor over the months given, held in every one of those months. The weights use the
    months they are applied to, which no investor could have done. No cap is applied: the Table 5 note
    limits the sum of the absolute weights on the two factors to 5, and gamma = 5 keeps these weights
    far below it. The scale of the managed factor (its c) changes x_sigma and leaves the returns as
    they are. Returns (the monthly return series, the weights)."""
    both = pd.concat([managed.rename("managed"), original.rename("original")], axis=1).dropna()
    w = mean_variance_weights(both.to_numpy(), gamma)
    return pd.Series(both.to_numpy() @ w, index=both.index, name="s3"), w


def percentiles(s: pd.Series, levels=(5, 50, 95)) -> dict:
    """The given percentiles of a series and its maximum, keyed P5, P50, ..., max."""
    out = {f"P{p}": float(np.percentile(s.dropna(), p)) for p in levels}
    out["max"] = float(s.max())
    return out

In [ ]:
%%writefile src/sq/regression.py
"""
The regression of a volatility-managed factor on its original factor, and the
measures the papers report from it.

    f_sigma_t = alpha + beta f_t + e_t, monthly, by ordinary least squares.

Two standard errors are computed for every coefficient:

  White (1980), the heteroskedasticity-robust standard error, which allows the
  size of e_t to change from month to month. Moreira and Muir's Table I and
  Cederburg et al.'s Table 3 use it (constants.SE_E1); E1 is judged with it.

  Newey and West (1987), which also allows e_t to be correlated with the
  previous L months' e, with weights 1 - l / (L + 1) on lag l (the Bartlett
  kernel), L = constants.NEWEY_WEST_LAGS. It is reported beside White's and is
  the one E4 to E6 use.

Neither applies a small-sample correction, which is the textbook form of both.

Units follow Moreira and Muir's tables (Table I note, p. 1619): alpha, its
standard error and the RMSE (the standard deviation of e) are multiplied by 12
and given in per cent a year; the appraisal ratio is alpha / RMSE times the
square root of 12 (footnote 11, p. 1620); Sharpe ratios are the monthly mean
over the monthly standard deviation times the square root of 12.

Two Sharpe ratios are compared with Jobson and Korkie's test, as Cederburg et
al. compare them (Table 1, p. 99, and Table 5, p. 106).
"""
from __future__ import annotations

from statistics import NormalDist

import numpy as np
import pandas as pd

from . import constants as C


def ols(y: np.ndarray, x: np.ndarray, nw_lags: int = C.NEWEY_WEST_LAGS) -> dict:
    """Regression of y on a constant and x. Returns the coefficients, both standard errors, the residuals,
    N, R2 and the RMSE, all monthly and in decimals."""
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(y)), np.asarray(x, dtype=float)])
    n, k = X.shape
    xtx_inv = np.linalg.inv(X.T @ X)
    coef = xtx_inv @ X.T @ y
    e = y - X @ coef
    xe = X * e[:, None]                                  # row t is x_t e_t
    meat_white = xe.T @ xe
    meat_nw = meat_white.copy()
    for lag in range(1, nw_lags + 1):
        w = 1 - lag / (nw_lags + 1)
        g = xe[lag:].T @ xe[:-lag]
        meat_nw += w * (g + g.T)
    se_white = np.sqrt(np.diag(xtx_inv @ meat_white @ xtx_inv))
    se_nw = np.sqrt(np.diag(xtx_inv @ meat_nw @ xtx_inv))
    ss_res = float(e @ e)
    ss_tot = float(((y - y.mean()) ** 2).sum())
    return {
        "alpha": coef[0], "beta": coef[1],
        "alpha_se_white": se_white[0], "beta_se_white": se_white[1],
        "alpha_se_nw": se_nw[0], "beta_se_nw": se_nw[1],
        "residuals": e, "N": n, "R2": 1 - ss_res / ss_tot,
        "rmse": np.sqrt(ss_res / (n - k)),
    }


def sharpe(r: pd.Series | np.ndarray) -> float:
    """The annualised Sharpe ratio of a monthly excess return: mean over standard deviation times the
    square root of 12."""
    r = np.asarray(r, dtype=float)
    return float(r.mean() / r.std(ddof=1) * C.SHARPE_ANNUALISE)


def jobson_korkie(r_i: pd.Series | np.ndarray, r_j: pd.Series | np.ndarray) -> dict:
    """Jobson and Korkie's (1981) test that two return series over the same months have the same Sharpe
    ratio, with the correction of Memmel (2003), as Cederburg et al. write it (p. 99):

        z = (s_j m_i - s_i m_j) / sqrt(theta),
        theta = (1 / T) (2 s_i^2 s_j^2 - 2 s_i s_j s_ij + m_i^2 s_j^2 / 2 + m_j^2 s_i^2 / 2
                         - (m_i m_j / (s_i s_j)) s_ij^2),

    m the monthly means, s the monthly standard deviations, s_ij the covariance, T the months. z is
    compared with the normal distribution. Returns the annualised Sharpe ratio difference i minus j, z
    and the two-sided p-value."""
    r_i = np.asarray(r_i, dtype=float)
    r_j = np.asarray(r_j, dtype=float)
    if len(r_i) != len(r_j):
        raise ValueError("the two series must cover the same months")
    t = len(r_i)
    m_i, m_j = r_i.mean(), r_j.mean()
    s_i, s_j = r_i.std(ddof=1), r_j.std(ddof=1)
    s_ij = float(np.cov(r_i, r_j)[0, 1])
    theta = (2 * s_i**2 * s_j**2 - 2 * s_i * s_j * s_ij + 0.5 * m_i**2 * s_j**2 + 0.5 * m_j**2 * s_i**2
             - (m_i * m_j / (s_i * s_j)) * s_ij**2) / t
    z = (s_j * m_i - s_i * m_j) / np.sqrt(theta)
    p = 2 * (1 - NormalDist().cdf(abs(z)))
    return {"difference": (m_i / s_i - m_j / s_j) * C.SHARPE_ANNUALISE, "z": float(z), "p": float(p)}


def table_i_row(managed: pd.Series, original: pd.Series) -> dict:
    """One column of Moreira and Muir's Table I, in the paper's units, with both standard errors."""
    fit = ols(managed.to_numpy(), original.to_numpy())
    a = C.ALPHA_ANNUALISE * 100
    return {
        "beta": fit["beta"], "beta_se_white": fit["beta_se_white"],
        "alpha": fit["alpha"] * a, "alpha_se_white": fit["alpha_se_white"] * a, "alpha_se_nw": fit["alpha_se_nw"] * a,
        "N": fit["N"], "R2": fit["R2"], "RMSE": fit["rmse"] * a,
        "appraisal": fit["alpha"] / fit["rmse"] * C.SHARPE_ANNUALISE,
        "sharpe_managed": sharpe(managed), "sharpe_original": sharpe(original),
    }

In [ ]:
%%writefile src/sq/verdicts.py
"""
The expectations judged, one function each, from the computed and the
published numbers. Each returns a table with one row per comparison and a
verdict that reads the levels from constants.py and nothing else.
"""
from __future__ import annotations

from statistics import NormalDist

import pandas as pd

from . import constants as C
from . import published as P

# The two-sided 5 per cent line for a t-statistic, from the normal distribution: 1.96.
Z_5 = NormalDist().inv_cdf(1 - C.SIGNIFICANCE_LEVEL / 2)


def e1(computed: pd.DataFrame) -> tuple[pd.DataFrame, bool]:
    """E1 on Moreira and Muir's sample. `computed` has one row per factor with the columns of
    regression.table_i_row. Returns the comparison table, one row per factor, and whether E1 is met."""
    rows = []
    for f in C.FACTORS:
        beta, beta_se, alpha_p, se_p, n_p, r2_p, rmse_p = P.MM_TABLE_I[f]
        c = computed.loc[f]
        t_p = alpha_p / se_p
        t_c = c["alpha"] / c["alpha_se_white"]
        ar_p = P.mm_appraisal_from_table_i(f)
        row = {
            "factor": f,
            "alpha published": alpha_p, "alpha computed": c["alpha"],
            "se published": se_p, "se computed (White)": c["alpha_se_white"], "se computed (Newey-West)": c["alpha_se_nw"],
            "t published": t_p, "t computed (White)": t_c,
            "appraisal published": ar_p, "appraisal computed": c["appraisal"],
            "N published": n_p, "N computed": int(c["N"]),
            "sign matches": (alpha_p > 0) == (c["alpha"] > 0),
            "5% category matches": (abs(t_p) > Z_5) == (abs(t_c) > Z_5),
            "alpha within one published se": abs(c["alpha"] - alpha_p) <= C.E1_ALPHA_WITHIN_SE * se_p,
            "appraisal within 0.05": abs(c["appraisal"] - ar_p) <= C.E1_APPRAISAL_BAND,
        }
        legs = ["sign matches", "5% category matches", "alpha within one published se", "appraisal within 0.05"]
        if f == "Mkt-RF":
            row["Sharpe published (Table V)"] = P.MM_TABLE_V_MARKET["sharpe"]
            row["Sharpe computed"] = c["sharpe_managed"]
            row["Sharpe within 0.05"] = abs(c["sharpe_managed"] - P.MM_TABLE_V_MARKET["sharpe"]) <= C.E1_MARKET_SHARPE_BAND
            legs.append("Sharpe within 0.05")
        row["factor matches"] = all(row[leg] for leg in legs)
        rows.append(row)
    table = pd.DataFrame(rows).set_index("factor")
    met = int(table["factor matches"].sum()) >= C.E1_MIN_FACTORS
    return table, met


def _outside(computed: float, published: float, band: float) -> float:
    """How far a computed number lies outside the band around the published one (0 when inside)."""
    return max(0.0, abs(computed - published) - band)


def e2(computed: pd.DataFrame) -> tuple[pd.DataFrame, dict, bool]:
    """E2 on Cederburg et al.'s sample. `computed` has one row per factor and the columns sharpe_original,
    sharpe_managed (Table 1), s1, s2, s1_minus_s2, p_s1_s2, s3, s1_minus_s3 (Table 5, Panel A), Sharpe
    ratios annualised. Returns the comparison table (one row per factor, four comparisons each), the
    result of each clause (i) to (iv), and whether E2 is met: every clause must hold."""
    band, band_s3 = C.E2_SHARPE_BAND, C.E2_S1_S3_BAND
    rows = []
    for f in C.FACTORS:
        c = computed.loc[f]
        o_p, m_p = P.COWY_TABLE_1_ORIGINAL[f][2], P.COWY_TABLE_1_MANAGED[f][2]
        s1_p, s2_p, d12_p, s3_p, d13_p = P.COWY_TABLE_5[f]
        p12_p = P.COWY_TABLE_5_P[f][0]
        significant = c["p_s1_s2"] < C.SIGNIFICANCE_LEVEL
        row = {
            "factor": f,
            "original SR published": o_p, "original SR computed": c["sharpe_original"],
            "managed SR published": m_p, "managed SR computed": c["sharpe_managed"],
            "(i) both within 0.05": abs(c["sharpe_original"] - o_p) <= band and abs(c["sharpe_managed"] - m_p) <= band,
            "S1 published": s1_p, "S1 computed": c["s1"], "S2 published": s2_p, "S2 computed": c["s2"],
            "(ii) both within 0.05": abs(c["s1"] - s1_p) <= band and abs(c["s2"] - s2_p) <= band,
            "S1-S2 published": d12_p, "S1-S2 computed": c["s1_minus_s2"],
            "p published": p12_p, "p computed": c["p_s1_s2"],
            "sign matches": (d12_p > 0) == (c["s1_minus_s2"] > 0),
            "significant at 5%": significant,
            "significance as published": significant == (f in C.E2_ONLY_SIGNIFICANT),
            "S1-S3 published": d13_p, "S1-S3 computed": c["s1_minus_s3"],
            "(iv) within 0.05": abs(c["s1_minus_s3"] - d13_p) <= band_s3,
        }
        row["(iii) sign and significance as published"] = row["sign matches"] and row["significance as published"]
        row["comparisons holding"] = int(row["(i) both within 0.05"]) + int(row["(ii) both within 0.05"]) \
            + int(row["(iii) sign and significance as published"]) + int(row["(iv) within 0.05"])
        rows.append(row)
    table = pd.DataFrame(rows).set_index("factor")
    significant_set = set(table.index[table["significant at 5%"]])
    clauses = {
        "(i)": bool(table["(i) both within 0.05"].all()),
        "(ii)": bool(table["(ii) both within 0.05"].all()),
        "(iii)": int(table["sign matches"].sum()) >= C.E2_SIGN_MIN_FACTORS and significant_set == set(C.E2_ONLY_SIGNIFICANT),
        "(iv)": bool(table["(iv) within 0.05"].all()),
    }
    return table, clauses, all(clauses.values())


def e2_line(table: pd.DataFrame, clauses: dict) -> str:
    """The verdict of E2 in words, with the arithmetic beside each clause."""
    band = C.E2_SHARPE_BAND
    n_hold, n_all = int(table["comparisons holding"].sum()), 4 * len(table)
    met = all(clauses.values())
    parts = [f"E2: {'MET' if met else 'NOT MET'}. {n_hold} of {n_all} comparisons hold."]

    def misses(pairs):
        out = []
        for f in table.index:
            for label, comp, pub in pairs:
                c, p = table.loc[f, comp], table.loc[f, pub]
                gap = _outside(c, p, band)
                if gap > 0:
                    out.append(f"{f}'s {label} {c:.3f} against {p:.2f} published, {gap:.3f} outside the band of {band}")
        return out

    m1 = misses([("original Sharpe ratio", "original SR computed", "original SR published"),
                 ("managed Sharpe ratio", "managed SR computed", "managed SR published")])
    parts.append("(i) met: every original and managed Sharpe ratio within 0.05 of Table 1." if clauses["(i)"]
                 else "(i) not met: " + "; ".join(m1) + ".")
    m2 = misses([("S1", "S1 computed", "S1 published"), ("S2", "S2 computed", "S2 published")])
    parts.append("(ii) met: every S1 and S2 within 0.05 of Table 5." if clauses["(ii)"]
                 else "(ii) not met: " + "; ".join(m2) + ".")
    n_sign = int(table["sign matches"].sum())
    sig = ", ".join(table.index[table["significant at 5%"]]) or "none"
    parts.append(f"(iii) {'met' if clauses['(iii)'] else 'not met'}: the sign of S1 minus S2 matches for {n_sign} of "
                 f"{len(table)}, at least {C.E2_SIGN_MIN_FACTORS} required; significant at 5 per cent: {sig}; "
                 f"published: {', '.join(C.E2_ONLY_SIGNIFICANT)}.")
    off = [f for f in table.index if not table.loc[f, "(iv) within 0.05"]]
    parts.append("(iv) met: S1 minus S3 within 0.05 of Table 5 for all six." if clauses["(iv)"]
                 else "(iv) not met: S1 minus S3 outside 0.05 for " + ", ".join(off) + ".")
    return "\n".join(parts)

In [ ]:
import sys
sys.path.insert(0, "src")
import numpy as np
import pandas as pd
from sq import constants as C, published as P, french_loader as fl, managed as M, regression as R, verdicts as V

records = [fl.download(key) for key in C.FRENCH_FILES]
fl.write_provenance(records)
VINTAGE = sorted({r["vintage_line"].split("the ")[-1].split(" ")[0] for r in records})[-1]
print("vintage:", VINTAGE)
monthly = fl.factors("monthly")
daily = fl.factors("daily")
pd.set_option("display.width", 200)

vm = {}
for f in C.FACTORS:
    rv = M.rv_mm(daily[f].dropna())               # Moreira and Muir's definition, equation 2
    vm[f], _ = M.volatility_managed(monthly[f], rv, C.FACTOR_FIRST_MONTH[f], C.MM_SAMPLE_END)
print({f: len(vm[f]) for f in C.FACTORS})

## 2. The regressions

Each column regresses the volatility-managed factor on the original factor over the factor's months on Moreira and Muir's sample, in the paper's units.

In [ ]:
computed = pd.DataFrame({f: R.table_i_row(vm[f]["managed"], vm[f]["f"]) for f in C.FACTORS}).T
print(computed[["beta", "beta_se_white", "alpha", "alpha_se_white", "N", "R2", "RMSE", "appraisal"]].astype(float).round(3).T.to_string())

## 3. The market, from monthly numbers to the paper's units

The regression runs on monthly decimal returns. Table I multiplies alpha, its standard error and the RMSE by 12 and writes them in per cent; the appraisal ratio, alpha over RMSE, is then a monthly ratio, and footnote 11 multiplies it by the square root of 12 to make it yearly.

In [ ]:
fit = R.ols(vm["Mkt-RF"]["managed"].to_numpy(), vm["Mkt-RF"]["f"].to_numpy())
print("monthly alpha:", round(fit["alpha"], 6), " times 12, in per cent:", round(fit["alpha"] * 1200, 3))
print("monthly standard error (White):", round(fit["alpha_se_white"], 6), " times 12, in per cent:", round(fit["alpha_se_white"] * 1200, 3))
print("monthly RMSE:", round(fit["rmse"], 6), " times 12, in per cent:", round(fit["rmse"] * 1200, 3))
print("appraisal ratio: alpha / RMSE * sqrt(12) =", round(fit["alpha"] / fit["rmse"] * np.sqrt(12), 4))
print("Sharpe ratio, volatility-managed:", round(R.sharpe(vm["Mkt-RF"]["managed"]), 4), " original:", round(R.sharpe(vm["Mkt-RF"]["f"]), 4))

## 4. Beside the published numbers, and E1 judged

The first table puts the numbers E1 judges beside the paper's, with each comparison's result. The second gives the numbers E1 does not judge (beta, R² and RMSE) beside the paper's, for the reader.

In [ ]:
e1_table, e1_met = V.e1(computed)
print(e1_table.T.to_string())
print()
others = pd.DataFrame({
    "beta published": {f: P.MM_TABLE_I[f][0] for f in C.FACTORS}, "beta computed": computed["beta"].astype(float),
    "R2 published": {f: P.MM_TABLE_I[f][5] for f in C.FACTORS}, "R2 computed": computed["R2"].astype(float),
    "RMSE published": {f: P.MM_TABLE_I[f][6] for f in C.FACTORS}, "RMSE computed": computed["RMSE"].astype(float),
})
print(others.round(3).T.to_string())
print()
n_match = int(e1_table["factor matches"].sum())
print(f"E1: {'MET' if e1_met else 'NOT MET'}. {n_match} of 6 factors match on every comparison; at least {C.E1_MIN_FACTORS} required.")

## 5. White and Newey-West standard errors

E1 uses White's standard error because Table I does. The study's later tests (E4 to E6) use Newey and West's, which also allows a month's residual to be correlated with the 12 months before it. Where the residuals of neighbouring months move together, a run of months carries less independent information than its count suggests, and the Newey-West standard error is larger.

In [ ]:
se = computed[["alpha", "alpha_se_white", "alpha_se_nw"]].astype(float)
se["t White"] = se["alpha"] / se["alpha_se_white"]
se["t Newey-West"] = se["alpha"] / se["alpha_se_nw"]
se["Newey-West over White"] = se["alpha_se_nw"] / se["alpha_se_white"]
print(se.round(3).to_string())

## 6. The outputs, the summary, and what could be wrong

The computed Table I, the E1 comparison and the verdict go to `outputs/` with the vintage in each name.

In [ ]:
computed.to_csv(f"outputs/mm_table_i_computed_{VINTAGE}.csv")
e1_table.to_csv(f"outputs/e1_comparison_{VINTAGE}.csv")
pd.DataFrame([{"expectation": "E1", "factors matching": n_match, "required": C.E1_MIN_FACTORS,
               "verdict": "met" if e1_met else "not met", "vintage": VINTAGE}]).to_csv(f"outputs/e1_verdict_{VINTAGE}.csv", index=False)

summary = pd.DataFrame([{
    "expectation": "E1: Moreira and Muir's Table I on their sample",
    "result": f"{n_match} of 6 factors match on sign, 5% category, alpha within one se, appraisal ratio within 0.05"
              f"; market Sharpe ratio {computed.loc['Mkt-RF', 'sharpe_managed']:.3f} against 0.52",
    "verdict": "met" if e1_met else "not met",
}]).set_index("expectation")
print(summary.to_string())
print(sorted(os.listdir("outputs")))

What could be wrong, and where each point is tested:

- **French's data have been revised since the paper.** A computed alpha can differ from the published one with the code right. The size of the revision is the difference between the two columns; the comparison of the same construction on Cederburg et al.'s later download (their Table 1, to December 2016) in notebook 04 gives a second reading on a second vintage.
- **The signal's definition.** Moreira and Muir's equation 2 is written for a 22-day month and the module removes the month's own mean (notebook 02). The match of the RMSE and the appraisal ratio is the test: a different signal would change the managed series and both numbers with it.
- **Significance rests on a t-statistic, which assumes the estimate is close to normally distributed.** Notebook 06 tests the same gain with Ledoit and Wolf's bootstrap, which does not assume it, and E4 asks whether the two verdicts agree for at least five of the six factors.
- **The alpha is a property of the regression.** It says the volatility-managed factor earned more than its beta times the original factor over these months. Whether an investor could have captured it is the subject of notebook 04, and whether it survives trading costs that of notebook 08.